# Equity Data Explorer — AAPL, USLM, JPM

Side-by-side walkthrough of every silver and gold table for three companies that span very different business profiles:

- **AAPL** — mega-cap consumer tech, fiscal year ends September, high-margin
- **USLM** — small-cap industrial (lime & limestone), fiscal year ends December, cyclical
- **JPM** — money-center bank, fiscal year ends December, very different statement structure (no COGS, leverage as a feature)

Use this notebook to: (1) see the shape of the data, (2) sanity-check what's clean vs. messy, (3) brainstorm analysis questions you want to support in the eventual parameterized notebook.

All scratchpad questions go in the last cell. Add to it as ideas come.

## 0. Setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Suppress the InsecureRequestWarning that comes from verify=False on R2 calls
warnings.filterwarnings("ignore", message=".*Unverified HTTPS request.*")

# Load .env from project root (one level up from notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

# Make pandas show wide tables in full
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Tickers under the microscope
TICKERS = ["AAPL", "USLM", "JPM"]
TICKERS_SQL = "(" + ", ".join(f"'{t}'" for t in TICKERS) + ")"

BUCKET = os.environ["R2_BUCKET_NAME"]
ACCT = os.environ["R2_ACCOUNT_ID"]
AK = os.environ["R2_ACCESS_KEY_ID"]
SK = os.environ["R2_SECRET_ACCESS_KEY"]

def open_con() -> duckdb.DuckDBPyConnection:
    c = duckdb.connect()
    c.execute("INSTALL httpfs; LOAD httpfs;")
    c.execute(f"SET s3_endpoint = '{ACCT}.r2.cloudflarestorage.com';")
    c.execute(f"SET s3_access_key_id = '{AK}';")
    c.execute(f"SET s3_secret_access_key = '{SK}';")
    c.execute("SET s3_region = 'auto';")
    c.execute("SET s3_url_style = 'path';")
    return c

def silver(path: str) -> str:
    return f"read_parquet('s3://{BUCKET}/silver/{path}', union_by_name=true)"

def gold(path: str) -> str:
    return f"read_parquet('s3://{BUCKET}/gold/{path}', union_by_name=true)"

con = open_con()
print("DuckDB connected. Tickers:", TICKERS)

## 1. Company snapshot — `silver/dim_company`

The static identity record for each ticker. Sector, industry, market cap, fiscal year end. Useful to keep nearby as a reminder of what business you're looking at.

In [ ]:
df = con.execute(f"""
    SELECT *
    FROM {silver('dim_company/*.parquet')}
    WHERE symbol IN {TICKERS_SQL}
    ORDER BY symbol
""").df()
df.T  # transpose so columns become rows (easier to read side-by-side)

## 2. Daily prices — `silver/fact_daily_prices`

Raw OHLCV + adjusted close. **Use `adjusted_close` for any return calculation** — it retroactively corrects for splits and dividends.

In [ ]:
# Row counts + date range per ticker
con.execute(f"""
    SELECT symbol,
           COUNT(*) AS rows,
           MIN(trade_date) AS first_date,
           MAX(trade_date) AS last_date
    FROM {silver('fact_daily_prices/**/*.parquet')}
    WHERE symbol IN {TICKERS_SQL}
    GROUP BY symbol
    ORDER BY symbol
""").df()

In [ ]:
# Last 5 trading days per ticker
con.execute(f"""
    SELECT *
    FROM (
        SELECT symbol, trade_date, open, high, low, close, adjusted_close, volume,
               dividend_amount, split_coefficient,
               ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY trade_date DESC) AS rn
        FROM {silver('fact_daily_prices/**/*.parquet')}
        WHERE symbol IN {TICKERS_SQL}
    )
    WHERE rn <= 5
    ORDER BY symbol, trade_date DESC
""").df()

In [ ]:
# Price chart — adjusted close, normalized to 100 at the start of the visible window
# (so radically different price levels can be compared)
px = con.execute(f"""
    SELECT symbol, trade_date, adjusted_close
    FROM {silver('fact_daily_prices/**/*.parquet')}
    WHERE symbol IN {TICKERS_SQL}
      AND trade_date >= DATE '2015-01-01'
    ORDER BY symbol, trade_date
""").df()

wide = px.pivot(index='trade_date', columns='symbol', values='adjusted_close')
normed = wide.divide(wide.iloc[0]) * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
wide.plot(ax=axes[0], title='Adjusted close (raw $)')
axes[0].set_ylabel('USD')
normed.plot(ax=axes[1], title='Adjusted close (rebased to 100 at start)')
axes[1].set_ylabel('Index (100 = 2015-01-01)')
axes[1].axhline(100, color='gray', linestyle=':', linewidth=0.8)
plt.tight_layout()
plt.show()

## 3. Income statement — `silver/fact_income_statement`

Both annual and quarterly rows in the same table, distinguished by `period_type`. Note: **no EPS column** — AV's income statement endpoint doesn't return it. EPS lives in `fact_earnings`.

Compare statement *shapes* across the three companies: AAPL has heavy R&D and a clean gross-profit structure; USLM is a simpler industrial; JPM is a bank — no COGS in the traditional sense, lots of interest line items, very different income structure.

In [ ]:
# Most recent 6 annual income statements per ticker
con.execute(f"""
    SELECT symbol, fiscal_date_ending,
           total_revenue, cost_of_revenue, gross_profit,
           operating_expenses, operating_income, ebit, ebitda,
           depreciation_amortization,
           income_before_tax, income_tax, net_income,
           net_income_continuing_ops,
           r_and_d, sga, interest_expense
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY fiscal_date_ending DESC) AS rn
        FROM {silver('fact_income_statement/*.parquet')}
        WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
    )
    WHERE rn <= 6
    ORDER BY symbol, fiscal_date_ending DESC
""").df()

In [ ]:
# Latest 8 quarterly income statements per ticker — feel the cadence
con.execute(f"""
    SELECT symbol, fiscal_date_ending,
           total_revenue, gross_profit, operating_income, net_income
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY fiscal_date_ending DESC) AS rn
        FROM {silver('fact_income_statement/*.parquet')}
        WHERE symbol IN {TICKERS_SQL} AND period_type = 'quarterly'
    )
    WHERE rn <= 8
    ORDER BY symbol, fiscal_date_ending DESC
""").df()

In [ ]:
# Revenue + net income trend — annual
df = con.execute(f"""
    SELECT symbol, fiscal_date_ending, total_revenue, net_income
    FROM {silver('fact_income_statement/*.parquet')}
    WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
      AND fiscal_date_ending >= DATE '2005-01-01'
    ORDER BY symbol, fiscal_date_ending
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for sym in TICKERS:
    s = df[df.symbol == sym]
    axes[0].plot(s.fiscal_date_ending, s.total_revenue / 1e9, marker='o', label=sym)
    axes[1].plot(s.fiscal_date_ending, s.net_income / 1e9, marker='o', label=sym)
axes[0].set_title('Annual revenue ($B)')
axes[1].set_title('Annual net income ($B)')
for ax in axes:
    ax.set_xlabel('Fiscal year end'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Balance sheet — `silver/fact_balance_sheet`

In [ ]:
# Latest 4 annuals — full picture
con.execute(f"""
    SELECT *
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY fiscal_date_ending DESC) AS rn
        FROM {silver('fact_balance_sheet/*.parquet')}
        WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
    )
    WHERE rn <= 4
    ORDER BY symbol, fiscal_date_ending DESC
""").df()

## 5. Cash flow — `silver/fact_cash_flow`

`free_cash_flow = operating_cashflow - abs(capex)` is the one derived field permitted in silver. Everything else here is direct from AV.

In [ ]:
con.execute(f"""
    SELECT symbol, fiscal_date_ending,
           operating_cashflow, capex, free_cash_flow,
           dividend_payout, repurchase_of_stock,
           investing_cashflow, financing_cashflow, change_in_cash
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY fiscal_date_ending DESC) AS rn
        FROM {silver('fact_cash_flow/*.parquet')}
        WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
    )
    WHERE rn <= 6
    ORDER BY symbol, fiscal_date_ending DESC
""").df()

In [ ]:
# Operating cash flow vs FCF — annual
df = con.execute(f"""
    SELECT symbol, fiscal_date_ending, operating_cashflow, free_cash_flow
    FROM {silver('fact_cash_flow/*.parquet')}
    WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
      AND fiscal_date_ending >= DATE '2010-01-01'
    ORDER BY symbol, fiscal_date_ending
""").df()

fig, axes = plt.subplots(1, len(TICKERS), figsize=(16, 4), sharey=False)
for ax, sym in zip(axes, TICKERS):
    s = df[df.symbol == sym]
    ax.bar(s.fiscal_date_ending.astype(str), s.operating_cashflow / 1e9, alpha=0.5, label='OpCF')
    ax.bar(s.fiscal_date_ending.astype(str), s.free_cash_flow / 1e9, alpha=0.9, label='FCF')
    ax.set_title(f'{sym} — annual cash flow ($B)')
    ax.legend(); ax.tick_params(axis='x', rotation=45); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Earnings — `silver/fact_earnings`

Both annual (just `reported_eps`) and quarterly (with consensus estimate + surprise + report date).

**Use `report_date` not `fiscal_date_ending`** when relating earnings to market reactions — fiscal_date_ending is when the quarter ended, report_date is when results were made public.

In [ ]:
# Last 8 quarterly earnings — surprise history
con.execute(f"""
    SELECT symbol, fiscal_date_ending, report_date,
           reported_eps, estimated_eps, surprise, surprise_pct
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY fiscal_date_ending DESC) AS rn
        FROM {silver('fact_earnings/*.parquet')}
        WHERE symbol IN {TICKERS_SQL} AND period_type = 'quarterly'
    )
    WHERE rn <= 8
    ORDER BY symbol, fiscal_date_ending DESC
""").df()

In [ ]:
# Annual EPS history — confirms the silver fix worked (no spurious quarter-end rows)
con.execute(f"""
    SELECT symbol, fiscal_date_ending, reported_eps
    FROM {silver('fact_earnings/*.parquet')}
    WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
      AND fiscal_date_ending >= DATE '2015-01-01'
    ORDER BY symbol, fiscal_date_ending DESC
""").df()

## 7. Corporate actions — `silver/fact_dividends` & `silver/fact_splits`

In [ ]:
# Recent dividends — note USLM is a regular payer, AAPL is too, JPM is too (all 3 dividend payers, useful)
con.execute(f"""
    SELECT *
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY ex_date DESC) AS rn
        FROM {silver('fact_dividends/*.parquet')}
        WHERE symbol IN {TICKERS_SQL}
    )
    WHERE rn <= 8
    ORDER BY symbol, ex_date DESC
""").df()

In [ ]:
# Splits — AAPL has a famous history of splits; USLM and JPM less so
con.execute(f"""
    SELECT *
    FROM {silver('fact_splits/*.parquet')}
    WHERE symbol IN {TICKERS_SQL}
    ORDER BY symbol, effective_date DESC
""").df()

## 8. GOLD — `gold/fact_fundamentals_wide`

The good stuff. One row per (symbol, fiscal_date_ending, period_type) with:
- All 4 silver statements joined
- Margins, ROE/ROA/ROIC, leverage, quality ratios
- YoY growth and QoQ revenue growth
- TTM aggregates on quarterly rows
- 3y / 5y EPS CAGR (constant per symbol, sourced from latest annual)

In [ ]:
# Column inventory — see what's available
cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {gold('fact_fundamentals_wide/*.parquet')}").fetchall()]
print(f"{len(cols)} columns in fact_fundamentals_wide:")
for c in cols:
    print(f"  - {c}")

In [ ]:
# Latest 4 quarters per ticker — wide "company snapshot" view
snap = con.execute(f"""
    SELECT symbol, fiscal_date_ending, period_type,
           total_revenue, total_revenue_ttm,
           net_income, net_income_ttm,
           free_cash_flow, free_cash_flow_ttm,
           reported_eps, reported_eps_ttm,
           gross_margin, operating_margin, net_margin, fcf_margin,
           roe, roa, roic,
           debt_to_equity, net_debt,
           revenue_growth_yoy, eps_growth_yoy, fcf_growth_yoy,
           eps_cagr_5y, eps_cagr_3y, eps_cagr_as_of
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY symbol, period_type ORDER BY fiscal_date_ending DESC) AS rn
        FROM {gold('fact_fundamentals_wide/*.parquet')}
        WHERE symbol IN {TICKERS_SQL} AND period_type = 'quarterly'
    )
    WHERE rn <= 4
    ORDER BY symbol, fiscal_date_ending DESC
""").df()
snap

In [ ]:
# Latest CAGR per company
con.execute(f"""
    SELECT DISTINCT symbol,
           ROUND(eps_cagr_5y * 100, 1) AS eps_cagr_5y_pct,
           ROUND(eps_cagr_3y * 100, 1) AS eps_cagr_3y_pct,
           eps_cagr_as_of
    FROM {gold('fact_fundamentals_wide/*.parquet')}
    WHERE symbol IN {TICKERS_SQL}
    ORDER BY symbol
""").df()

## 9. Margin trends (gold)

Compare margin trajectories. Watch for:
- AAPL: hardware → services mix shifting → gross margin uptrend
- USLM: cyclical margins tracking commodity prices and demand
- JPM: "net_margin" for a bank is conceptually different — driven by net interest margin + non-interest income

In [ ]:
df = con.execute(f"""
    SELECT symbol, fiscal_date_ending,
           gross_margin, operating_margin, net_margin, fcf_margin
    FROM {gold('fact_fundamentals_wide/*.parquet')}
    WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
      AND fiscal_date_ending >= DATE '2010-01-01'
    ORDER BY symbol, fiscal_date_ending
""").df()

fig, axes = plt.subplots(1, len(TICKERS), figsize=(16, 4), sharey=True)
for ax, sym in zip(axes, TICKERS):
    s = df[df.symbol == sym]
    for col, label in [('gross_margin', 'Gross'),
                        ('operating_margin', 'Operating'),
                        ('net_margin', 'Net'),
                        ('fcf_margin', 'FCF')]:
        if col in s.columns:
            ax.plot(s.fiscal_date_ending, s[col] * 100, marker='o', label=label)
    ax.set_title(f'{sym} — annual margins (%)')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 10. Growth dashboard (gold)

Year-over-year growth on revenue, EPS, FCF — annual rows.

In [ ]:
df = con.execute(f"""
    SELECT symbol, fiscal_date_ending,
           revenue_growth_yoy, eps_growth_yoy, fcf_growth_yoy
    FROM {gold('fact_fundamentals_wide/*.parquet')}
    WHERE symbol IN {TICKERS_SQL} AND period_type = 'annual'
      AND fiscal_date_ending >= DATE '2010-01-01'
    ORDER BY symbol, fiscal_date_ending
""").df()

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, col, title in zip(axes,
                           ['revenue_growth_yoy', 'eps_growth_yoy', 'fcf_growth_yoy'],
                           ['Revenue YoY growth (%)', 'EPS YoY growth (%)', 'FCF YoY growth (%)']):
    for sym in TICKERS:
        s = df[df.symbol == sym]
        ax.plot(s.fiscal_date_ending, s[col] * 100, marker='o', label=sym)
    ax.set_title(title); ax.axhline(0, color='gray', linewidth=0.5)
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. TTM trends (gold quarterly)

TTM (trailing twelve months) smooths out seasonality. Each quarterly row carries `<metric>_ttm` = sum of the last 4 quarters. Better for trend-watching than raw quarterly numbers.

In [ ]:
df = con.execute(f"""
    SELECT symbol, fiscal_date_ending,
           total_revenue_ttm, net_income_ttm, free_cash_flow_ttm, reported_eps_ttm
    FROM {gold('fact_fundamentals_wide/*.parquet')}
    WHERE symbol IN {TICKERS_SQL} AND period_type = 'quarterly'
      AND fiscal_date_ending >= DATE '2015-01-01'
      AND total_revenue_ttm IS NOT NULL
    ORDER BY symbol, fiscal_date_ending
""").df()

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
metrics = [('total_revenue_ttm', 'TTM revenue ($B)', 1e9),
           ('net_income_ttm',    'TTM net income ($B)', 1e9),
           ('free_cash_flow_ttm','TTM free cash flow ($B)', 1e9),
           ('reported_eps_ttm',  'TTM EPS ($)', 1)]
for ax, (col, title, scale) in zip(axes.flat, metrics):
    for sym in TICKERS:
        s = df[df.symbol == sym]
        ax.plot(s.fiscal_date_ending, s[col] / scale, marker='.', label=sym)
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Elfenbein heuristic preview

Using the gold layer's `eps_cagr_5y` and the latest TTM EPS we can do a quick Elfenbein fair P/E and fair price calc by hand (this will be a proper column in `fact_valuation_daily` once Phase B lands). Recall:

$$\text{Fair P/E} = \frac{\text{Growth Rate \%}}{2} + 8$$
$$\text{Fair Price} = \text{Fair P/E} \times \text{forward (or TTM) EPS}$$

Margin of safety: trade at 30%+ below fair price.

In [ ]:
snap = con.execute(f"""
    WITH latest_q AS (
        SELECT symbol, reported_eps_ttm, eps_cagr_5y, eps_cagr_3y,
               ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY fiscal_date_ending DESC) AS rn
        FROM {gold('fact_fundamentals_wide/*.parquet')}
        WHERE symbol IN {TICKERS_SQL}
          AND period_type = 'quarterly'
          AND reported_eps_ttm IS NOT NULL
    ),
    latest_px AS (
        SELECT symbol, trade_date, adjusted_close,
               ROW_NUMBER() OVER (PARTITION BY symbol ORDER BY trade_date DESC) AS rn
        FROM {silver('fact_daily_prices/**/*.parquet')}
        WHERE symbol IN {TICKERS_SQL}
    )
    SELECT l.symbol,
           ROUND(p.adjusted_close, 2) AS current_price,
           ROUND(l.reported_eps_ttm, 2) AS eps_ttm,
           ROUND(l.eps_cagr_5y * 100, 1) AS growth_5y_pct,
           ROUND(l.eps_cagr_5y * 100 / 2 + 8, 2) AS fair_pe_elfenbein,
           ROUND((l.eps_cagr_5y * 100 / 2 + 8) * l.reported_eps_ttm, 2) AS fair_price,
           ROUND(((l.eps_cagr_5y * 100 / 2 + 8) * l.reported_eps_ttm - p.adjusted_close)
                  / p.adjusted_close * 100, 1) AS upside_pct,
           p.adjusted_close <= 0.7 * (l.eps_cagr_5y * 100 / 2 + 8) * l.reported_eps_ttm AS mos_30pct_met
    FROM latest_q l
    JOIN latest_px p USING (symbol)
    WHERE l.rn = 1 AND p.rn = 1
    ORDER BY l.symbol
""").df()
snap

## 13. Gold tables not yet built locally

These will exist after the next scheduled workflow runs:
- `gold/fact_prices_enriched/year=*/...` — built daily after `transform_daily_prices`
- `gold/dim_company_enriched/dim_company_enriched.parquet` — built weekly after `build_fundamentals_wide`

To build them right now without waiting, run from a project-root terminal:

```bash
python -m transform_gold.build_prices_enriched
python -m transform_gold.build_dim_company_enriched
```

Once they exist, uncomment and run the cell below to take a look.

In [ ]:
# # Recent rows from fact_prices_enriched — once built
# con.execute(f"""
#     SELECT symbol, trade_date, adjusted_close,
#            return_1d, return_21d, return_252d,
#            sma_50, sma_200, rsi_14,
#            bb_upper, bb_lower, bb_pct_b,
#            rel_strength_vs_spy_3m, rel_strength_vs_spy_12m,
#            pct_off_52w_high
#     FROM {gold('fact_prices_enriched/**/*.parquet')}
#     WHERE symbol IN {TICKERS_SQL}
#       AND trade_date >= DATE '2026-05-01'
#     ORDER BY symbol, trade_date DESC
# """).df()

In [ ]:
# # dim_company_enriched — once built
# con.execute(f"""
#     SELECT * FROM {gold('dim_company_enriched/*.parquet')}
#     WHERE symbol IN {TICKERS_SQL}
# """).df().T

## 14. Scratchpad — questions for the parameterized notebook

Capture analysis questions and ideas here as you scroll through the cells above. Each one is a candidate cell or section in the eventual parameterized `analyze_ticker.ipynb`.

**Identity / context**
- [ ] What does this company *do*? (sector, industry, country, FY end, listing date)
- [ ] How big is it now vs. historical peak? (mcap, revenue)

**Price action**
- [ ] Total return 1y / 3y / 5y / 10y
- [ ] Total return vs SPY over each window
- [ ] Where is the stock vs its 52w / 5y / all-time high?
- [ ] Current technical posture (above/below 50d & 200d? overbought RSI? Bollinger band position?)
- [ ] Has there been a recent breakout or breakdown?

**Operating performance**
- [ ] Multi-year revenue trend — accelerating, steady, decelerating?
- [ ] Margin trend — expanding, stable, compressing? Where vs sector?
- [ ] Quality of earnings: net income vs operating cash flow vs FCF — are they tracking?
- [ ] Capital intensity (capex as % revenue, depreciation as % revenue)
- [ ] R&D as % of revenue — and trend

**Capital allocation**
- [ ] Buybacks vs dividends vs reinvestment over the last 5 years
- [ ] Net debt trend — adding or paying down
- [ ] Dividend history — coverage ratio, growth rate

**Returns on capital**
- [ ] ROE / ROA / ROIC trend
- [ ] Above or below sector median?

**Earnings dynamics**
- [ ] Beat-vs-miss history — typical surprise %
- [ ] EPS revision trajectory (estimated vs reported over time)
- [ ] Post-earnings price reaction (T+1, T+5 returns after each report)

**Valuation**
- [ ] P/E TTM, P/S TTM, EV/EBITDA TTM — vs historical range, vs sector
- [ ] Elfenbein fair P/E vs current P/E
- [ ] FCF yield
- [ ] Dividend yield (and is it covered)

**Comparison mode (Seeking Alpha-style)**
- [ ] Pick a peer set — show all metrics side-by-side
- [ ] Highlight where the company is best / worst in cohort
- [ ] Ranked tables for each metric

**Risks / red flags to surface automatically**
- [ ] Margin compression over last 3 quarters
- [ ] Cash flow diverging from net income (low cash conversion)
- [ ] Rising debt without rising EBITDA
- [ ] Negative or rapidly decelerating growth
- [ ] Earnings miss + EPS revision lower
- [ ] Trading near 52w low
- [ ] Goodwill that's a big % of equity (write-down risk)

Add your own as you go...